In [24]:
!pip install -q gpytorch transformers tqdm scikit-learn torchvision

import pandas as pd, torch, torch.nn as nn, numpy as np, tqdm, gpytorch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from PIL import Image
from transformers import ViTForImageClassification, ViTConfig, AutoImageProcessor
from torchvision import transforms
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ExponentialLR

# CONFIG
pretrained_vit_dir = '/kaggle/input/models/pretrained_vit/pretrained_vit'
eurosat_input = '/kaggle/input/eurosat-dataset/EuroSAT'
num_classes = 10; low_dim = 10; batch_size = 16; n_epochs = 100
lr_vit = 1e-8; lr_gp = 1e-3; patience = 10; test_size_val = 0.2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = True; WARMUP_EPOCHS = 5

# DATA
train_df = pd.read_csv(f'{eurosat_input}/train.csv')
test_df = pd.read_csv(f'{eurosat_input}/test.csv')
classes = sorted(train_df['Label'].unique())
label_map = {c:i for i,c in enumerate(classes)}

train_subset = pd.concat([train_df[train_df['Label']==c].sample(20, random_state=42) for c in classes])
image_dir = eurosat_input + '/'
train_subset['image_path'] = image_dir + train_subset['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

train_sub, val_sub = train_test_split(train_subset, test_size=test_size_val, stratify=train_subset['Label'], random_state=42)
train_sub['label'] = train_sub['Label'].map(label_map)
val_sub['label'] = val_sub['Label'].map(label_map)
test_df['label'] = test_df['Label'].map(label_map)

# DATASET
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
aug = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                          transforms.RandomRotation(90), transforms.ColorJitter(.2,.2,.2,.1)])

class EuroSATDataset(Dataset):
    def __init__(self, df, proc, aug=False): self.df = df.reset_index(drop=True); self.proc = proc; self.aug = aug
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]['image_path']).convert('RGB')
        if self.aug: img = aug(img)
        x = self.proc(img, return_tensors='pt')['pixel_values'].squeeze(0)
        return x, self.df.iloc[i]['label']

train_ds = EuroSATDataset(train_sub, processor, True)
val_ds   = EuroSATDataset(val_sub, processor)
test_ds  = EuroSATDataset(test_df, processor)

train_loader = DataLoader(train_ds, batch_size, True)
val_loader   = DataLoader(val_ds, batch_size)
test_loader  = DataLoader(test_ds, batch_size)
num_train = len(train_ds)

# ViT
cfg = ViTConfig.from_pretrained(pretrained_vit_dir)
vit = ViTForImageClassification.from_pretrained(pretrained_vit_dir, config=cfg)
feature_extractor = vit.vit.to(device)
for p in feature_extractor.parameters(): p.requires_grad = False

# PROJECTION + PCA
scaler = GradScaler(enabled=use_amp)
projection = nn.Linear(768, low_dim).to(device)

print("PCA init...")
feats = []
with torch.no_grad():
    for x,_ in train_loader:
        x = x.to(device)
        with autocast('cuda', enabled=use_amp):
            out = feature_extractor(x)
            f = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:,0]
        feats.append(f.cpu())
feats = torch.cat(feats, 0)

mean = feats.mean(0, keepdim=True)
centered = feats - mean
U,S,Vh = torch.linalg.svd(centered, full_matrices=False)
with torch.no_grad():
    projection.weight.data = Vh[:low_dim].to(device)
    projection.bias.data = (-mean.to(device) @ projection.weight.data.T).squeeze(0)

proj_feats = []
with torch.no_grad():
    for i in range(0,len(feats),batch_size):
        b = feats[i:i+batch_size].to(device)
        proj_feats.append(projection(b).cpu())
proj_feats = torch.cat(proj_feats,0).numpy()

# INDUCING POINTS
num_inducing = min(30, num_train)
kmeans = MiniBatchKMeans(n_clusters=num_inducing, n_init='auto', random_state=42)
kmeans.fit(proj_feats)
inducing_points = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32).to(device)
inducing_points += 1e-5 * torch.randn_like(inducing_points)

# GP LAYER (RBF + HARD LOCK)
class GPLayer(gpytorch.models.ApproximateGP):
    def __init__(self, Z, ncls, dim):
        Z = Z + 1e-5 * torch.randn_like(Z)
        var_dist = gpytorch.variational.CholeskyVariationalDistribution(Z.size(0), batch_shape=torch.Size([ncls]))
        var_strat = gpytorch.variational.IndependentMultitaskVariationalStrategy(
            gpytorch.variational.VariationalStrategy(self, Z, var_dist, learn_inducing_locations=True),
            num_tasks=ncls
        )
        super().__init__(var_strat)
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([ncls]))
        base = gpytorch.kernels.RBFKernel(ard_num_dims=dim, batch_shape=torch.Size([ncls]))

        # SAFE MEDIAN HEURISTIC
        with torch.no_grad():
            if Z.size(0) > 1:
                d = torch.cdist(Z, Z)
                median = torch.median(d[d>0]).item()
            else:
                median = 1.0
            ls = median * 0.7
            base.lengthscale = ls
        base.lengthscale_constraint = gpytorch.constraints.Interval(ls*0.9, ls*1.1)

        self.covar_module = gpytorch.kernels.ScaleKernel(
            base, batch_shape=torch.Size([ncls]),
            outputscale_constraint=gpytorch.constraints.Interval(0.5,3.0)
        )
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))

# DKL MODEL
class DKLModel(nn.Module):
    def __init__(self, fe, ld, nc, Z):
        super().__init__()
        self.feature_extractor = fe
        self.projection = projection
        self.gp_layer = GPLayer(Z, nc, ld)
    def forward(self, x):
        out = self.feature_extractor(x)
        f = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:,0]
        p = self.projection(f)
        p = (p - p.mean(0,keepdim=True)) / (p.std(0,keepdim=True)+1e-5)
        return self.gp_layer(p.float())

# MODEL
model = DKLModel(feature_extractor, low_dim, num_classes, inducing_points).to(device).float()
likelihood = gpytorch.likelihoods.SoftmaxLikelihood(num_classes=num_classes, num_features=num_classes).to(device).float()

# OPTIM
optimizer = AdamW([
    {'params': model.projection.parameters(),      'lr': lr_gp},
    {'params': model.gp_layer.hyperparameters(),   'lr': lr_gp},
    {'params': model.gp_layer.variational_parameters(), 'lr': lr_gp},
    {'params': likelihood.parameters(),            'lr': lr_gp}
])
scheduler = ExponentialLR(optimizer, gamma=0.99)
mll = gpytorch.mlls.VariationalELBO(likelihood, model.gp_layer, num_data=num_train)

# TRAIN / EVAL
def train_epoch(ep):
    model.train(); likelihood.train(); loss_sum = 0
    for x,y in tqdm.tqdm(train_loader, desc=f'Epoch {ep}'):
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=use_amp), gpytorch.settings.cholesky_jitter(1e-1):
            out = model(x)
            loss = -mll(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()
    return loss_sum/len(train_loader)

def evaluate(loader, name='Val'):
    model.eval(); likelihood.eval(); correct = total = 0
    with torch.no_grad(), gpytorch.settings.fast_pred_var(), gpytorch.settings.num_likelihood_samples(64), gpytorch.settings.cholesky_jitter(1e-1):
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = likelihood(model(x))
            pred = out.probs.mean(0).argmax(-1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)
    acc = correct/total
    print(f'{name} Acc: {acc:.4f}')
    return acc

# TRAINING LOOP
best_val = 0; patience_cnt = 0
for ep in range(1, n_epochs+1):
    if ep == WARMUP_EPOCHS+1:
        print(f'--- Epoch {ep}: UNFREEZING ViT ---')
        for p in model.feature_extractor.parameters(): p.requires_grad = True
        optimizer = AdamW([
            {'params': model.feature_extractor.parameters(), 'lr': lr_vit},
            {'params': model.projection.parameters(),      'lr': lr_gp},
            {'params': model.gp_layer.hyperparameters(),   'lr': lr_gp},
            {'params': model.gp_layer.variational_parameters(),'lr': lr_gp},
            {'params': likelihood.parameters(),            'lr': lr_gp}
        ])
        scheduler = ExponentialLR(optimizer, gamma=0.99)
    loss = train_epoch(ep)
    scheduler.step()
    print(f'Epoch {ep}  Loss:{loss:.4f}')
    val_acc = evaluate(val_loader)
    if val_acc > best_val:
        best_val = val_acc
        torch.save({'model':model.state_dict(), 'likelihood':likelihood.state_dict()}, 'best_dkl.pt')
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= patience:
            print('Early stop')
            break

# FINAL TEST
ckpt = torch.load('best_dkl.pt', map_location=device)
model.load_state_dict(ckpt['model'])
likelihood.load_state_dict(ckpt['likelihood'])
evaluate(test_loader, 'Test')

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


PCA init...


Epoch 1: 100%|██████████| 10/10 [00:01<00:00,  7.44it/s]


Epoch 1  Loss:2.3942
Val Acc: 0.1250


Epoch 2: 100%|██████████| 10/10 [00:01<00:00,  8.22it/s]


...

Epoch 20: 100%|██████████| 10/10 [00:02<00:00,  4.43it/s]


Epoch 20  Loss:2.3098
Val Acc: 0.2000


Epoch 21: 100%|██████████| 10/10 [00:02<00:00,  4.38it/s]


Epoch 21  Loss:2.3056
Val Acc: 0.2250
Early stop
Test Acc: 0.1437


0.1437037037037037

In [25]:
# --------------------------------------------------------------
# FINAL — LINEAR KERNEL, EXACT GP, 20-SHOT (FULLY WORKING)
# --------------------------------------------------------------
!pip install -q gpytorch transformers tqdm scikit-learn torchvision

import pandas as pd, torch, torch.nn as nn, numpy as np, gpytorch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from PIL import Image
from transformers import ViTForImageClassification, ViTConfig, AutoImageProcessor
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ExponentialLR

# -------------------------- CONFIG --------------------------
pretrained_vit_dir = '/kaggle/input/models/pretrained_vit/pretrained_vit'
eurosat_input      = '/kaggle/input/eurosat-dataset/EuroSAT'
num_classes        = 10
low_dim            = 10
batch_size         = 16
n_epochs           = 60
lr_vit             = 1e-8      # after warm-up
lr_proj            = 1e-3
lr_gp              = 5e-2      # <<-- increased
patience           = 12
test_size_val      = 0.2
device             = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp            = True
WARMUP_EPOCHS      = 5
# -----------------------------------------------------------

# --------------------------- DATA ---------------------------
train_df = pd.read_csv(f'{eurosat_input}/train.csv')
test_df  = pd.read_csv(f'{eurosat_input}/test.csv')
classes  = sorted(train_df['Label'].unique())
label_map = {c:i for i,c in enumerate(classes)}

# 20-shot per class
train_subset = pd.concat([train_df[train_df['Label']==c].sample(20, random_state=42) for c in classes])
image_dir = eurosat_input + '/'
train_subset['image_path'] = image_dir + train_subset['Filename']
test_df['image_path']      = image_dir + test_df['Filename']

train_sub, val_sub = train_test_split(train_subset, test_size=test_size_val,
                                      stratify=train_subset['Label'], random_state=42)

train_sub['label'] = train_sub['Label'].map(label_map)
val_sub['label']   = val_sub['Label'].map(label_map)
test_df['label']   = test_df['Label'].map(label_map)

# ----------------------- DATASET ---------------------------
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224', use_fast=True)
aug = transforms.Compose([transforms.RandomHorizontalFlip(),
                          transforms.RandomVerticalFlip(),
                          transforms.RandomRotation(90),
                          transforms.ColorJitter(.2,.2,.2,.1)])

class EuroSATDataset(Dataset):
    def __init__(self, df, proc, augment=False):
        self.df = df.reset_index(drop=True); self.proc = proc; self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]['image_path']).convert('RGB')
        if self.augment: img = aug(img)
        x = self.proc(img, return_tensors='pt')['pixel_values'].squeeze(0)
        return x, self.df.iloc[i]['label']

train_ds = EuroSATDataset(train_sub, processor, augment=True)
val_ds   = EuroSATDataset(val_sub,   processor, augment=False)
test_ds  = EuroSATDataset(test_df,    processor, augment=False)

train_loader = DataLoader(train_ds, batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size, shuffle=False)

# --------------------------- ViT ---------------------------
cfg = ViTConfig.from_pretrained(pretrained_vit_dir)
vit = ViTForImageClassification.from_pretrained(pretrained_vit_dir, config=cfg)
feature_extractor = vit.vit.to(device)
for p in feature_extractor.parameters(): p.requires_grad = False   # frozen at start

# ------------------- RANDOM PROJECTION INIT ----------------
print("Random projection initialisation ...")
projection = nn.Linear(768, low_dim, bias=True).to(device)
nn.init.kaiming_normal_(projection.weight, mode='fan_in', nonlinearity='linear')
nn.init.zeros_(projection.bias)

proj_train = []
with torch.no_grad():
    for x, _ in train_loader:
        x = x.to(device)
        with autocast('cuda', enabled=use_amp):
            out = feature_extractor(x)
            f = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:,0]
            p = projection(f)
        proj_train.append(p.cpu())
proj_train = torch.cat(proj_train, 0)

proj_mean = proj_train.mean(0, keepdim=True)
proj_std  = proj_train.std(0, keepdim=True) + 1e-5
proj_train = (proj_train - proj_mean) / proj_std
proj_train = proj_train.to(device)

# --------------------- LABELS & SCALED TARGETS -------------
train_targets = torch.tensor(train_sub['label'].values, dtype=torch.long).to(device)

SCALE = 10.0                                 # <<-- crucial scaling
train_y_onehot = torch.nn.functional.one_hot(train_targets, num_classes).float()
train_y_onehot = (train_y_onehot - 0.5) * SCALE      # now -5 … +5
train_y_onehot = train_y_onehot.to(device)

# ----------------------- LINEAR GP -------------------------
class LinearGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y):
        likelihood = gpytorch.likelihoods.GaussianLikelihood()
        super().__init__(train_x, train_y, likelihood)

        self.mean = gpytorch.means.ConstantMean()
        self.mean.constant = torch.tensor(0.1, device=device)   # small positive bias

        base = gpytorch.kernels.LinearKernel(ard_num_dims=low_dim)
        self.covar = gpytorch.kernels.ScaleKernel(
            base,
            outputscale_constraint=gpytorch.constraints.Interval(2.0, 30.0)
        )
        self.covar.outputscale = 8.0          # start large

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean(x), self.covar(x))

# -------------------------- DKL MODEL ----------------------
class DKLModel(nn.Module):
    def __init__(self, fe, train_x, train_y_onehot, proj_mean, proj_std):
        super().__init__()
        self.feature_extractor = fe
        self.projection = projection
        self.gps = nn.ModuleList([
            LinearGP(train_x, train_y_onehot[:, i]) for i in range(num_classes)
        ])
        self.train_x   = train_x
        self.proj_mean = proj_mean.to(device)
        self.proj_std  = proj_std.to(device)

    def forward(self, x):
        if x is None:                                 # training → cached features
            projected = self.train_x
        else:
            out = self.feature_extractor(x)
            f = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:,0]
            p = self.projection(f)
            projected = (p - self.proj_mean) / self.proj_std
        return torch.stack([gp(projected).mean for gp in self.gps], dim=-1)

model = DKLModel(feature_extractor, proj_train, train_y_onehot, proj_mean, proj_std).to(device)

# -------------------------- OPTIMIZER ----------------------
optimizer = AdamW([
    {'params': model.projection.parameters(), 'lr': lr_proj},
    {'params': [p for gp in model.gps for p in gp.parameters()], 'lr': lr_gp}
], weight_decay=1e-4)
scheduler = ExponentialLR(optimizer, gamma=0.99)
scaler = GradScaler(enabled=use_amp)

# ----------------------- TRAIN / EVAL ----------------------
def train_epoch(ep):
    model.train()
    optimizer.zero_grad()
    with autocast('cuda', enabled=use_amp), gpytorch.settings.cholesky_jitter(1e-1):
        logits = model(None)                                 # (N, C)
        loss = torch.nn.functional.cross_entropy(logits, train_targets)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss.item()

def evaluate(loader, name='Val'):
    model.eval()
    correct = total = 0
    with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-1):
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits = model(x)
            pred = logits.argmax(-1)
            correct += pred.eq(y).sum().item()
            total   += y.size(0)
    acc = correct / total
    print(f'{name} Acc: {acc:.4f}')
    return acc

# -------------------------- TRAINING LOOP -----------------
best_val = 0.0
patience_cnt = 0

for ep in range(1, n_epochs+1):
    if ep == WARMUP_EPOCHS + 1:
        print(f'--- Epoch {ep}: UNFREEZING ViT (lr={lr_vit}) ---')
        for p in model.feature_extractor.parameters(): p.requires_grad = True
        optimizer = AdamW([
            {'params': model.feature_extractor.parameters(), 'lr': lr_vit},
            {'params': model.projection.parameters(),      'lr': lr_proj},
            {'params': [p for gp in model.gps for p in gp.parameters()], 'lr': lr_gp}
        ], weight_decay=1e-4)
        scheduler = ExponentialLR(optimizer, gamma=0.99)

    loss = train_epoch(ep)
    scheduler.step()
    print(f'Epoch {ep:02d}  Loss: {loss:.4f}')

    val_acc = evaluate(val_loader, 'Val')
    if val_acc > best_val:
        best_val = val_acc
        torch.save({'model': model.state_dict()}, 'best_dkl.pt')
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= patience:
            print('Early stopping triggered')
            break

# -------------------------- FINAL TEST --------------------
ckpt = torch.load('best_dkl.pt', map_location=device)
model.load_state_dict(ckpt['model'])
_ = evaluate(test_loader, 'Test')

Random projection initialisation ...
Epoch 01  Loss: 2.3026
Val Acc: 0.0250
Epoch 02  Loss: 2.3026
Val Acc: 0.0250
Epoch 03  Loss: 2.3031
Val Acc: 0.0250
Epoch 04  Loss: 2.3026
Val Acc: 0.0250
Epoch 05  Loss: 2.3028
Val Acc: 0.0250
--- Epoch 6: UNFREEZING ViT (lr=1e-08) ---
Epoch 06  Loss: 2.3029
Val Acc: 0.0250
Epoch 07  Loss: 2.3028
Val Acc: 0.0250
Epoch 08  Loss: 2.3028
Val Acc: 0.0250
Epoch 09  Loss: 2.3026
Val Acc: 0.0250
Epoch 10  Loss: 2.3026
Val Acc: 0.0250
Epoch 11  Loss: 2.3027
Val Acc: 0.0250
Epoch 12  Loss: 2.3027
Val Acc: 0.0250
Epoch 13  Loss: 2.3026
Val Acc: 0.0250
Early stopping triggered
Test Acc: 0.1000
